# Project Starter: Option 2, Restaurant Reviews (Yelp Open Dataset)

**Data:** reviews from the Yelp Open Dataset, which you download yourself after you accept Yelp's terms. The helper script `project_starters/yelp_sample.py` draws a fixed sample from one city and one category, for example 20,000 restaurant reviews from Philadelphia. The dataset guide (`dataset_option_2_yelp_reviews.md`) explains the download.

This notebook sets up your project data the same way every time. It draws or loads your sample, records where the data came from, audits it, protects the test data and saves the outputs that Milestone 1 (MS1) asks for. It does not build your final method.

| Part | What it does |
|---|---|
| 1 Setup | Packages, folders and your project settings |
| 2 Sample and provenance | Draw the sample once from your download, check it, record where it came from |
| 3 Audit | Missing and repeated text, length, stars, businesses, dates, personal details, reading |
| 4 Preparation check | Apply a preparation rule and count what it changes |
| 5 Protected split | Record the test IDs, add a guard; for your own label, samples to label and how to load them |
| 6 Evaluation plan | Write down the plan before you see any results |
| 7 First baseline (for MS2) | A default guess and a simple model; off until MS2 |
| 8 Save | Save the outputs and the reproducibility record, then commit |

**How to use it.**

1. Download the Yelp Open Dataset and put the two files `yelp_academic_dataset_business.json` and `yelp_academic_dataset_review.json` in `project/data/private/yelp/` (the notebook makes this folder for you in Part 1).
2. Copy this file into your project folder, `project/notebooks/`, and give it a clear name, such as `01_data_audit_and_split.ipynb`.
3. Fill in the settings in Part 1, then run all cells from the top (**Restart**, then **Run All**). The first run reads the 5 GB review file once and takes a few minutes. Later runs load the saved sample in seconds. Rerunning is always safe: the notebook never overwrites your label files.
4. Cells marked **Your turn** are for your own work, with a notes cell below each. The notebook runs without them.
5. Commit the executed notebook with MS1. The MS1 brief lists everything else your repository must hold.

**Yelp's terms.** You may use the data for academic study only. Commit your sample to your private course repository, but never the Yelp download. Do not show review text in public: paraphrase it in your report and slides. Delete the data when the course ends. The dataset guide gives the details.

This starter has no points. The milestone rubrics in Canvas describe how the project is graded.

## Part 1 Setup

### 1.1 Packages

Select the `ban6403` kernel, then run the cell. The tools for the reproducibility record (hashes and package versions) are loaded at the end, in the last part.

In [ ]:
from pathlib import Path
import json
import re
import sys
import textwrap
import unicodedata
import pandas as pd
import matplotlib.pyplot as plt

### 1.2 The folders

Run this notebook from `project/notebooks/`. The repository is then two folders up, `../..`, and every path below starts from there. This notebook uses four places, and the cell makes them if they do not exist yet:

- `project/outputs/option2_setup/`: audit tables without text or IDs, the evaluation plan and the manifest. Commit them.
- `project/data/sample/`: your sample, its manifest and the ID lists (protected test IDs, dropped copies). Commit them to your private course repository, so the instructor can rerun your work. Each file must stay under 50 MB.
- `project/data/annotations/`: the label templates and your label files, if you make your own label. Commit them too.
- `project/data/private/yelp/`: the two Yelp files you downloaded. The repository's `.gitignore` keeps `project/data/private/` out of Git.

In [ ]:
REPO = Path("../..")   # this notebook runs from project/notebooks/, two folders below the repository
OUT = REPO / "project/outputs/option2_setup"
SAMPLE_DIR = REPO / "project/data/sample"
ANNOT = REPO / "project/data/annotations"
PRIVATE = REPO / "project/data/private"
# mkdir makes a folder; parents=True also makes missing parent folders; exist_ok=True skips a folder that exists.
for folder in [OUT, SAMPLE_DIR, ANNOT, PRIVATE / "yelp"]:
    folder.mkdir(parents=True, exist_ok=True)

### 1.3 Your project settings

Fill in these settings. Keep them short; the MS1 plan explains them in full.

| Setting | What to enter |
|---|---|
| `DECISION_OWNER` | The person or team who will act on your result, such as "Operations manager of a restaurant group" |
| `BUSINESS_QUESTION` | The decision in one sentence |
| `RAW_DIR` | The folder with the two Yelp JSON files. Keep the default unless your files are elsewhere. |
| `CITY`, `STATE` | One city in the dataset. Run Part 2.1 once to list the cities with the most businesses. |
| `CATEGORY` | One Yelp category, such as `"Restaurants"`, `"Coffee & Tea"` or `"Hotels & Travel"` |
| `N_REVIEWS` | How many reviews to sample. 20,000 is plenty and fits any laptop. |
| `START_DATE`, `END_DATE` | Optional date limits, such as `"2017-01-01"`, or `""` |
| `TARGET` | The column you will predict or describe. The default, `"stars_band"` (low 1–2, mid 3, high 4–5), only lets the notebook run. Use `"own_label"` if you will label your own target (Part 5.2). |
| `RUN_MS2_BASELINE` | Keep `False` for MS1. Set it to `True` for MS2; Part 7 then fits a first model. |

**The sample is drawn once.** The first run saves `yelp_sample.csv` and its manifest in `project/data/sample/`. Later runs load them. If you change `CITY`, `STATE`, `CATEGORY`, `N_REVIEWS`, the dates or `SEED` afterwards, the notebook stops, because a new sample means a new split. Change them only before MS1, and record why.

In [ ]:
DECISION_OWNER = ""
BUSINESS_QUESTION = ""
RAW_DIR = "project/data/private/yelp"
CITY = "Philadelphia"
STATE = "PA"
CATEGORY = "Restaurants"
N_REVIEWS = 20000
START_DATE = ""
END_DATE = ""
TARGET = "stars_band"     # a column of the data, or "own_label" for your own label (Part 5.2)
RUN_MS2_BASELINE = False  # True from MS2 on: Part 7 then fits a first model
SEED = 6403               # fixes every random draw, so each run gives the same samples

# OWN_LABEL is True when you label your own target.
OWN_LABEL = TARGET == "own_label"
SETTINGS = {"decision_owner": DECISION_OWNER, "business_question": BUSINESS_QUESTION, "city": CITY,
            "state": STATE, "category": CATEGORY, "n_reviews": N_REVIEWS, "start_date": START_DATE,
            "end_date": END_DATE, "target": TARGET, "run_ms2_baseline": RUN_MS2_BASELINE}

# Print each setting; {name:>18} right-aligns the name. An empty setting is marked CHECK.
for name, value in SETTINGS.items():
    if value in ("", []):
        value = "CHECK: not filled in yet"
    print(f"{name:>18}: {value}")

## Part 2 Draw the sample and record its provenance

### 2.1 Check the download and list the cities

The helper script `yelp_sample.py` sits in the `project_starters` folder; the cell imports it from there. If the Yelp business file is in `RAW_DIR`, the cell lists the cities with the most businesses in your category, so you can choose `CITY` and `STATE`. The listing reads only the small business file. Once your sample is saved, the notebook reruns without the download.

In [ ]:
# sys.path lists the folders where Python looks for modules; add project_starters/ to find the sampler.
sys.path.append(str(REPO / "project_starters"))
import yelp_sample

raw_dir = REPO / RAW_DIR   # a full path in RAW_DIR stays as it is
SAMPLE = SAMPLE_DIR / "yelp_sample.csv"
SAMPLE_MANIFEST = SAMPLE_DIR / "yelp_sample_manifest.json"
business_file = raw_dir / yelp_sample.BUSINESS_FILE
if business_file.exists():
    city_counts = yelp_sample.list_cities(business_file, CATEGORY, top=12)

### 2.2 Draw the sample once, or load it

The helper reads the review file line by line, keeps reviews of businesses in your city and category, and draws `N_REVIEWS` of them with your seed. It drops the reviewer ID. It also makes the `split` column: about 70% of reviews for `train`, 15% for `valid` and 15% for `test`, with **all reviews of one business in the same split**. A model is then never tested on a restaurant it saw in training.

The same download and settings always give the same sample, on any computer. The sample manifest records the settings, the counts and the SHA-256 hashes of the input files and of the sample.

In [ ]:
# The settings the sample was drawn with; a saved sample must match them.
wanted = {"city": CITY, "state": STATE, "category": CATEGORY, "start_date": None, "end_date": None}
if START_DATE:
    wanted["start_date"] = START_DATE
if END_DATE:
    wanted["end_date"] = END_DATE
if SAMPLE.exists() and SAMPLE_MANIFEST.exists():
    # Load the saved sample's manifest and compare its settings with yours.
    sample_manifest = json.loads(SAMPLE_MANIFEST.read_text())
    same = (sample_manifest["filters"] == wanted and sample_manifest["n_requested"] == N_REVIEWS
            and sample_manifest["seed"] == SEED)
    if not same:
        raise ValueError("Your settings differ from the saved sample. A new sample means a new split. "
                         "Restore the settings, or delete the sample and its manifest only before MS1, and record why.")
    print("Loaded the saved sample:", SAMPLE.name)
else:
    # First run: draw the sample once and save it with its manifest.
    sample_manifest = yelp_sample.make_sample(raw_dir, SAMPLE, CITY, STATE, CATEGORY, N_REVIEWS, SEED,
                                              START_DATE, END_DATE)
print(json.dumps(sample_manifest["counts"], indent=2))
# The file size in MB (stat().st_size is in bytes). GitHub warns above 50 MB.
size_mb = SAMPLE.stat().st_size / 1e6
print(f"{SAMPLE.name}: {size_mb:.1f} MB")
if size_mb >= 50:
    print("CHECK: above 50 MB. Draw a smaller sample before you commit it.")

In [ ]:
raw = pd.read_csv(SAMPLE, dtype={"review_id": str, "business_id": str})
data = raw.rename(columns={"review_id": "record_id"})
data["text"] = data["text"].fillna("")
data["date"] = pd.to_datetime(data["date"])
# pd.cut puts each star rating into a band: (0, 2] low, (2, 3] mid, (3, 5] high.
data["stars_band"] = pd.cut(data["stars"], [0, 2, 3, 5], labels=["low (1-2)", "mid (3)", "high (4-5)"]).astype(str)
print("reviews:", len(data), "| businesses:", data["business_id"].nunique(), "| columns:", list(raw.columns))
# Per split: agg counts the reviews (size) and the different businesses (nunique) in one table.
per_split = data.groupby("split").agg(reviews=("record_id", "size"), businesses=("business_id", "nunique"))
per_split.loc[["train", "valid", "test"]]

### 2.3 Provenance record

**Provenance** is the record of where data came from and what was done to it. Read this record, correct anything that does not fit, and copy it into your data dictionary (`project/docs/data_dictionary.md`). Fill in the date you downloaded the data.

In [ ]:
provenance = {
    "source": "Yelp Open Dataset, downloaded from Yelp's dataset page",
    "downloaded_on": "",   # the date you downloaded the files
    "terms": "Yelp Dataset Terms of Use: academic use only; no sharing of the data; no public display of reviews",
    "collection": "reviews written on Yelp by Yelp users; Yelp chose which businesses and reviews are in the dataset",
    "sample": f"{len(data):,} reviews of {data['business_id'].nunique():,} businesses; {CATEGORY} in {CITY}, {STATE}; seed {SEED}",
    "time_window": f"{data['date'].min().date()} to {data['date'].max().date()}",
    "unit_of_analysis": "one review of one business by one Yelp user",
    "stable_id": "review_id (assigned by Yelp)",
    "personal_details": "user_id dropped by the sampler; reviewer names are not in the review file; texts may name staff",
    "may_be_shared": "no: the download stays in project/data/private/; the sample is committed only to the private course repository",
}

# Print each entry; an empty one is marked CHECK.
for key, value in provenance.items():
    if value in ("", None):
        value = "CHECK: fill in"
    print(f"{key:>18}: {value}")

## Part 3 Audit the corpus

An audit answers one question: can this data support your decision? Each table below checks one risk.

### 3.1 Missing, short and repeated text

Exact copies matter most when they fall in different splits. A model could then be tested on text it has already seen. The **text key** ignores case, punctuation and extra spaces, in any alphabet, so near-copies count as copies. The column `unit` says what each row counts, and `of` gives the total of that unit.

In [ ]:
def text_key(text):
    """Take one text; return a key that ignores case, punctuation and extra spaces, in any alphabet."""
    # NFKC writes look-alike characters the same way; casefold() is lower() for every alphabet.
    text = unicodedata.normalize("NFKC", text).casefold()
    # \W+ is a run of characters that are not letters or digits; accented and non-Latin letters count as letters.
    return re.sub(r"\W+", " ", text).strip()

# map applies text_key to every text in the column.
data["text_key"] = data["text"].map(text_key)
# A text of only symbols or emoji has an empty key. Give it its own key, so such texts do not count as copies.
empty_key = data["text_key"].eq("")
data.loc[empty_key, "text_key"] = "empty-" + data.loc[empty_key, "record_id"]

# One row per check; "unit" says what the row counts and "of" is the total of that unit.
n_reviews = len(data)
n_keys = data["text_key"].nunique()
n_groups = data["business_id"].nunique()
# For each text key (or business): in how many different splits it appears.
splits_per_key = data.groupby("text_key")["split"].nunique()
splits_per_group = data.groupby("business_id")["split"].nunique()
audit = pd.DataFrame([
    {"check": "empty text", "count": int(data["text"].str.strip().eq("").sum()), "unit": "reviews", "of": n_reviews},
    {"check": "fewer than 10 words", "count": int(data["n_words"].lt(10).sum()), "unit": "reviews", "of": n_reviews},
    {"check": "text of only symbols or emoji", "count": int(empty_key.sum()), "unit": "reviews", "of": n_reviews},
    {"check": "repeated text: copies beyond the first", "count": int(data["text_key"].duplicated().sum()), "unit": "reviews", "of": n_reviews},
    {"check": "repeated texts found in more than one split", "count": int(splits_per_key.gt(1).sum()), "unit": "distinct texts", "of": n_keys},
    {"check": "record_id not unique", "count": int(data["record_id"].duplicated().sum()), "unit": "reviews", "of": n_reviews},
    {"check": "businesses in more than one split", "count": int(splits_per_group.gt(1).sum()), "unit": "businesses", "of": n_groups},
])

# Splits with fewer than 20 different businesses.
businesses_per_split = data.groupby("split")["business_id"].nunique()
few = {}
for split_name, n_businesses in businesses_per_split.items():
    if n_businesses < 20:
        few[split_name] = n_businesses
if few:
    print("CHECK: few businesses in", few, "- results there describe a handful of businesses. "
          "Consider a larger city or category before MS1.")
audit

### 3.2 Keep each repeated text in one split

A text that sits in two splits lets a method be tested on text it has already seen. The rule here: a copy stays in the earliest split, in the order `train`, `valid`, `test`. Later copies are left out of the project data. All reviews of one business still stay in one split; only the copies are dropped. The IDs of the dropped reviews go to `project/data/sample/dropped_cross_split_copies.csv`, so you can report them in your data section.

In [ ]:
# The text keys found in train, and those found in valid.
train_keys = set(data.loc[data["split"].eq("train"), "text_key"])
valid_keys = set(data.loc[data["split"].eq("valid"), "text_key"])
# isin is True where a key is in the set; & means "and", | means "or".
valid_copy = data["split"].eq("valid") & data["text_key"].isin(train_keys)
test_copy = data["split"].eq("test") & (data["text_key"].isin(train_keys) | data["text_key"].isin(valid_keys))
is_copy = valid_copy | test_copy
dropped_copies = data.loc[is_copy, ["record_id", "split"]]
dropped_copies.to_csv(SAMPLE_DIR / "dropped_cross_split_copies.csv", index=False)
# ~ means "not": keep the records that are not dropped copies.
data = data[~is_copy].copy()
print("copies dropped from valid:", int(valid_copy.sum()), "| from test:", int(test_copy.sum()),
      "| reviews left:", len(data))

### 3.3 Length, stars and dates by split

Yelp reviews vary a lot in length. Ratings lean toward 4 and 5 stars. The split by business can make the three splits differ a little; check that they are similar enough. `density=True` scales each histogram so that the area of its bars adds up to 1. Splits of different sizes can then be compared. A bar's share of reviews is its height times its width.

In [ ]:
# describe() gives the count, mean, quartiles (25%, 50% = median, 75%) and more; keep six of them.
length_table = data.groupby("split")["n_words"].describe()
length_table = length_table[["count", "min", "25%", "50%", "75%", "max"]].loc[["train", "valid", "test"]]
# normalize="columns" gives shares that add up to 1 in each split.
stars_table = pd.crosstab(data["stars"], data["split"], normalize="columns").round(3)[["train", "valid", "test"]]
data["year"] = data["date"].dt.year
# Reviews per year and split: size() counts the rows, and unstack() turns the splits into columns.
year_table = data.groupby(["year", "split"]).size().unstack(fill_value=0)
# reindex keeps all three split columns in order, with 0 when a split has no review that year.
year_table = year_table.reindex(columns=["train", "valid", "test"], fill_value=0)
print(length_table.round(1).to_string(), end="\n\n")
print("Share of reviews by stars (columns sum to 1):")
print(stars_table.to_string(), end="\n\n")
print("Reviews per year:")
print(year_table.tail(8).to_string())

In [ ]:
# One list of lengths per split; clip(upper=600) shows longer reviews as 600 words.
lengths = []
for split_name in ["train", "valid", "test"]:
    lengths.append(data.loc[data["split"].eq(split_name), "n_words"].clip(upper=600))
fig, ax = plt.subplots(figsize=(7, 3.2))
# density=True makes the bar areas of each split add up to 1; a bar's share is its height times its width.
ax.hist(lengths, bins=30, label=["train", "valid", "test"], density=True)
ax.set_xlabel("Words per review (600 or more shown as 600)")
ax.set_ylabel("Probability density")
ax.set_title("Review length by split")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "length_by_split.png", dpi=150)
plt.show()

### 3.4 Businesses: how concentrated is the sample?

A few popular businesses can hold many reviews. Then a result may describe those businesses more than the city. The table shows how many reviews the largest businesses hold.

In [ ]:
# Reviews per business, largest first: do a few businesses dominate the sample?
per_business = data.groupby("business_id").size().sort_values(ascending=False)
top10_reviews = int(per_business.head(10).sum())
concentration = pd.DataFrame([
    {"measure": "businesses in the sample", "value": len(per_business)},
    {"measure": "median reviews per business", "value": float(per_business.median())},
    {"measure": "reviews of the 10 largest businesses", "value": top10_reviews},
    {"measure": "share of reviews in the 10 largest businesses", "value": round(top10_reviews / len(data), 3)},
])
concentration

### 3.5 Personal details

The sampler dropped the reviewer ID, and the review file has no reviewer names. People still write details into their reviews: the names of staff, phone numbers, links. The scan counts texts that match common patterns. A match is only a lead: read some of them, and decide whether to mask the detail or drop the record. Write the decision in your data dictionary.

In [ ]:
# Patterns that may be personal details. A match is a reason to read the text, not proof.
# Pattern key: \w a letter or digit, \d a digit, \s a space, \b a word edge, {3} exactly three, + one or more.
PII_PATTERNS = {
    "email address": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone number": r"\(?\b\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}\b",
    "web link": r"https?://\S+|www\.\S+",
    "long digit string (card or account?)": r"\b\d{9,}\b",
    "@handle": r"(?<![\w.])@\w{3,}",
}

def pii_scan(texts):
    """Take a column of texts; return how many texts match each pattern above."""
    rows = []
    for name, pattern in PII_PATTERNS.items():
        # True for each text with at least one match.
        hits = texts.str.contains(pattern, regex=True)
        rows.append({"pattern": name, "texts with a match": int(hits.sum()), "of": len(texts)})
    return pd.DataFrame(rows)

pii_table = pii_scan(data["text"])
pii_table

### 3.6 Label balance

If `TARGET` is a column, this shows how many reviews have each value in each split. Small groups in `test` will give wide uncertainty intervals later.

In [ ]:
if OWN_LABEL:
    label_table = None
    display_label = "TARGET is your own label: Part 5.3 counts your labels."
elif TARGET in data.columns:
    print("reviews without a value:", int(data[TARGET].isna().sum()))
    # Records per label and split; unstack turns the splits into columns (0 where a label is missing).
    label_table = data.groupby([TARGET, "split"]).size().unstack(fill_value=0)
    label_table = label_table.reindex(columns=["train", "valid", "test"], fill_value=0)
    label_table["share of train"] = (label_table["train"] / label_table["train"].sum()).round(3)
    print("values:", len(label_table))
    display_label = label_table
else:
    label_table = None
    display_label = f"CHECK: TARGET '{TARGET}' is not a column of the data. Fix it in Part 1.3."
display_label

### 3.7 Read before you count

Numbers do not replace reading. The helper prints a few training records, drawn with your seed. The preview cuts each text, which is fine for browsing. When you judge a label, read the whole text, because a later sentence can change it: use `chars=None`.

In [ ]:
def show_examples(frame, n=3, seed=SEED, chars=300):
    """Take a table; print n records drawn with the seed, each cut to chars characters (chars=None: full text)."""
    examples = frame.sample(min(n, len(frame)), random_state=seed)
    # itertuples gives one row at a time; row.text is the value in the text column.
    for row in examples.itertuples(index=False):
        print(f"{row.record_id} | business {row.business_id} | {row.stars} stars | {row.split}")
        shown = row.text
        if chars is not None and len(shown) > chars:
            shown = shown[:chars] + "..."
        # textwrap.fill breaks the text into lines of at most 100 characters.
        print(textwrap.fill(shown, 100))
        print()

train_rows = data[data["split"].eq("train")]
show_examples(train_rows)

**Your turn A.** Read at least 15 training reviews in full. Note any personal details the scan missed and two cases that would be hard to label. Your course repository is private, so the output may stay. Do not copy review text into your report or slides: paraphrase it.

In [ ]:
# Your turn A
# TODO 1: draw 15 training reviews with another seed and print them in full:
#         show_examples(train_rows, n=15, seed=1, chars=None)
# TODO 2: write the IDs and your notes in the cell below (paraphrase; do not copy review text).

**Your notes (Your turn A)**

- Records read (IDs):
- Personal details the scan missed, and what you will do about them:
- Hard case 1 (ID, why it is hard):
- Hard case 2 (ID, why it is hard):

## Part 4 Preparation check

Preparation changes the text before a method sees it. Keep the raw text as your evidence, and record what the preparation rule changes. The steps run in order, and the table counts what each step changes, including the final space and lower-case steps. The rule below is a starting point: it replaces line breaks, links and phone numbers with plain tokens. Change it if your method needs something else, then run the notebook again.

In [ ]:
PREPARATION_STEPS = [
    ("line break -> space", r"\s*\n\s*", " "),
    ("web link -> URL", r"https?://\S+|www\.\S+", " URL "),
    ("phone number -> PHONE", r"\(?\b\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}\b", " PHONE "),
]

def prepare(text):
    """Take one text; return it after every preparation step, with single spaces and in lower case."""
    # Each step replaces its pattern with its token; _ skips the step name.
    for _, pattern, token in PREPARATION_STEPS:
        text = re.sub(pattern, token, text)
    # \s+ is any run of spaces or line breaks; it becomes one space. strip() removes spaces at both ends.
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

# The prepared text is what the model in Part 7 reads. The raw text stays in "text" as your evidence.
# map applies prepare to every text in the column.
data["text_prepared"] = data["text"].map(prepare)

# Count what each step changes. The steps are applied one after another, as in prepare.
rows = []
current = data["text"]
for name, pattern, token in PREPARATION_STEPS:
    # str.count gives the number of matches in each text; gt(0) is True for a text with at least one.
    hits = current.str.count(pattern)
    rows.append({"step": name, "reviews changed": int(hits.gt(0).sum()), "replacements": int(hits.sum()), "of": len(data)})
    current = current.str.replace(pattern, token, regex=True)
# The space step: runs of spaces become one space, and spaces at both ends go.
spaced = current.str.replace(r"\s+", " ", regex=True).str.strip()
rows.append({"step": "extra spaces -> one space, ends trimmed", "reviews changed": int(current.ne(spaced).sum()),
             "replacements": "", "of": len(data)})
# The last step: texts that change when they are written in lower case.
rows.append({"step": "lower case", "reviews changed": int(spaced.ne(spaced.str.lower()).sum()),
             "replacements": "", "of": len(data)})
prep_audit = pd.DataFrame(rows)
print(prep_audit.to_string(index=False))

The new column `text_prepared` is what the model in Part 7 reads. If you change the rule, run the notebook again, and the model sees the change. Keep quoting the raw text in your report.

## Part 5 Protect the test data

The test split is your final, honest check. You look at test results **once**, after you freeze your method in MS3. Until then you fit on `train` and compare on `valid` (with your own label: on your development sample only).

### 5.1 Record the test IDs and a guard

The test IDs go to `project/data/sample/protected_test_ids.csv`; their fingerprint goes into the manifest at the end. The guard `check_protected` stops a cell if a table you are about to fit or tune on holds a protected record. Protected means a test record, and, if you make your own label, a record of your held-out sample. Call the guard before every fit until MS3.

In [ ]:
test_ids = data.loc[data["split"].eq("test"), "record_id"].tolist()
# Save the test IDs, so you can show later that no method used them before MS3.
pd.DataFrame({"record_id": sorted(test_ids)}).to_csv(SAMPLE_DIR / "protected_test_ids.csv", index=False)
# A set of protected IDs; the held-out sample is added to it if you make your own label.
protected_ids = set(test_ids)

def check_protected(frame, id_col="record_id"):
    """Take a table; stop with an error if any of its IDs is protected (test, or your held-out sample)."""
    # & keeps the IDs that are in both sets.
    overlap = set(frame[id_col]) & protected_ids
    if overlap:
        raise ValueError(f"{len(overlap)} protected records found. Do not fit or tune on them before MS3.")
    print(f"ok: no protected records among {len(frame):,} rows")

print("protected test reviews:", len(test_ids))
check_protected(data[data["split"].ne("test")])

### 5.2 Samples to label (only for your own label)

Many projects need a label that the data does not have, such as "complains about the wait". Then set `TARGET = "own_label"` in Part 1.3, and read and label records yourself with your annotation guide. Every own-label project follows the same route:

- The **development sample** comes from `train`. Use it to write and test your guide, and to build and tune your methods.
- The **held-out sample** comes from `valid`. It is your evaluation set for MS3. Label it with the finished guide by the MS2 freeze, before any method sees it. With your own label, `valid` is not used for tuning, and the guard blocks the held-out records.
- Neither sample comes from `test`, so you never read a test record before MS3.

**How the files work.** The cell writes two blank templates to `project/data/annotations/`: `dev_labels_template.csv` and `heldout_labels_template.csv`. Copy each one to `dev_labels.csv` and `heldout_labels.csv`, and fill the `own_label` column in the copies. The notebook only reads these two files, so a rerun never erases your labels.

A template is written only when neither it nor your copy exists. To get a fresh template, for example after you change a sample size, delete the old template first. Do not use `NA` or `None` as a label value: pandas reads them as empty. Reading and labeling 100 short reviews takes most people one to two hours.

In [ ]:
N_DEV_LABEL = 100
N_HELDOUT_LABEL = 100
DEV_TEMPLATE = ANNOT / "dev_labels_template.csv"
HELDOUT_TEMPLATE = ANNOT / "heldout_labels_template.csv"
DEV_LABELS = ANNOT / "dev_labels.csv"           # your labeled copy: the notebook only reads it
HELDOUT_LABELS = ANNOT / "heldout_labels.csv"   # your labeled copy: the notebook only reads it

def label_sample(split, n):
    """Take a split name and a size; return a seeded sample of that split with empty own_label and note columns."""
    pool = data[data["split"].eq(split)]
    # min(n, len(pool)) never asks for more rows than the split has; random_state=SEED fixes the draw.
    sample = pool.sample(min(n, len(pool)), random_state=SEED)
    sample = sample[["record_id", "business_id", "stars", "text"]].sort_values("record_id")
    sample["own_label"] = ""
    sample["note"] = ""
    return sample

def write_template(sample, template, completed):
    """Save a blank template, but only if neither the template nor your completed file exists yet."""
    if template.exists() or completed.exists():
        print("kept as it is:", template.name, "and", completed.name)
    else:
        sample.to_csv(template, index=False)
        print("wrote", template.name)

heldout_ids = []
if OWN_LABEL:
    dev_sample = label_sample("train", N_DEV_LABEL)
    heldout_sample = label_sample("valid", N_HELDOUT_LABEL)
    write_template(dev_sample, DEV_TEMPLATE, DEV_LABELS)
    write_template(heldout_sample, HELDOUT_TEMPLATE, HELDOUT_LABELS)
    heldout_ids = heldout_sample["record_id"].tolist()
    # From now on the guard also blocks the held-out records.
    protected_ids.update(heldout_ids)
    print("development sample (train):", len(dev_sample), "| held-out sample (valid):", len(heldout_ids))
else:
    print(f"TARGET is the column '{TARGET}', so you do not need samples to label.")

### 5.3 Load your labels

Once you have labeled records in `dev_labels.csv`, this cell reads them and joins them to the data by `record_id`. A **join** (pandas `merge`) adds the columns of one table to the rows of another table that have the same ID. After the join, the column `own_label` holds your label for the records you labeled; it is empty for all others. The cell only counts the held-out labels. They stay unused until MS3.

In [ ]:
if OWN_LABEL and DEV_LABELS.exists():
    # Read the IDs as text, so the join matches them exactly (an ID such as 00123 keeps its zeros).
    dev_labels = pd.read_csv(DEV_LABELS, dtype={"record_id": str})
    # Keep the ID and the label, and drop the rows you have not labeled yet.
    dev_labels = dev_labels[["record_id", "own_label"]].dropna(subset=["own_label"])
    # merge joins the labels to data by ID; how="left" keeps every record of data, labeled or not.
    # drop(...) first removes an own_label column from an earlier run of this cell.
    data = data.drop(columns="own_label", errors="ignore").merge(dev_labels, on="record_id", how="left")
    labeled = data[data["own_label"].notna()]
    print("labeled development records:", len(labeled), "| outside train (should be 0):", int(labeled["split"].ne("train").sum()))
    print(labeled["own_label"].value_counts().to_string())
elif OWN_LABEL:
    print("No dev_labels.csv yet. Copy the template, label some records, then run the notebook again.")
if OWN_LABEL and HELDOUT_LABELS.exists():
    heldout_labels = pd.read_csv(HELDOUT_LABELS, dtype={"record_id": str})
    # The IDs in your held-out file are protected too, even if you later change the sample size.
    protected_ids.update(heldout_labels["record_id"])
    n_done = int(heldout_labels["own_label"].notna().sum())
    print(f"held-out labels filled in: {n_done} of {len(heldout_labels)} (finish them by the MS2 freeze)")

**Your turn B (own label only).** Label 20 reviews of the development sample with your draft guide. Read each text in full in the file or with `show_examples(..., chars=None)`. Note every case where the guide did not tell you what to do, then revise the guide. If your target is a column of the data, skip this.

In [ ]:
# Your turn B
# TODO 1: copy dev_labels_template.csv to dev_labels.csv (in project/data/annotations/).
# TODO 2: fill the own_label column for 20 records, and use the note column for doubts.
# TODO 3: run the notebook again; Part 5.3 then counts your labels.
# TODO 4: write the guide changes in the cell below and in project/docs/annotation_guide.md.

**Your notes (Your turn B)**

- Records labeled (IDs):
- Cases the guide did not cover (ID, what was unclear):
- Changes to the guide:

## Part 6 Write the evaluation plan now

Decide how you will judge success before you see any result. Then nobody, including you, can move the target after the fact. Fill in the plan. The cell flags any field that is still empty. The MS1 brief has a metric menu: a main metric and a baseline for each type of task, and the module that teaches it. The main metric and the success threshold are **provisional until MS2**: you may change them once, with a reason, when you freeze the final test plan.

The two data fields are filled in from your label route, so the saved plan always matches what the notebook does. Do not type over them.

In [ ]:
# The data roles follow your label route.
if OWN_LABEL:
    data_for_tuning = "own label: the labeled development sample from train only; valid is not used for tuning"
    data_for_final_check = (f"own label: the held-out sample from valid ({len(heldout_ids)} records), "
                            "labeled before the MS2 freeze and scored once in MS3")
else:
    data_for_tuning = "train to fit, valid to compare and tune"
    data_for_final_check = "test, scored once after the method is frozen in MS3"

EVALUATION_PLAN = {
    "task": "",               # for example: flag reviews that complain about service speed
    "output": "",             # what the decision owner receives
    "target": TARGET,
    "baseline": "",           # the simple method you must beat, for example: a keyword rule, then TF-IDF + logistic regression
    "candidate_methods": "",  # one or two methods you may compare in MS3
    "main_metric": "",        # see the metric menu in the MS1 brief
    "success_criterion": "",  # the business threshold (provisional until MS2)
    "data_for_tuning": data_for_tuning,
    "data_for_final_check": data_for_final_check,
    "uncertainty": "",        # for example: k of n now; a paired comparison with its bootstrap interval in MS3 (Lab 13)
}
# One row per field of the plan: ok, or CHECK when it is still empty.
check_rows = []
for field, value in EVALUATION_PLAN.items():
    if value.strip():
        status = "ok"
    else:
        status = "CHECK: empty"
    check_rows.append({"field": field, "status": status})
plan_check = pd.DataFrame(check_rows)
(OUT / "evaluation_plan.json").write_text(json.dumps(EVALUATION_PLAN, indent=2))
plan_check

## Part 7 A first baseline (for MS2)

You do not need a model for MS1, and classification is taught in Module 8. Keep `RUN_MS2_BASELINE = False` until then. For MS2, set it to `True` in Part 1.3. This part then shows whether your target can be predicted from the text.

- With a **column of the data** as the target, both methods are fitted on `train` and scored on `valid`.
- With **your own label**, they are fitted on 70% of your labeled development records and checked on the other 30%. Reviews of one business can fall in both parts, so read this score as a rough first check. Your held-out sample and `test` are never used.
- The **majority baseline** always guesses the most common training value.
- The **TF-IDF + logistic regression** pipeline is the course's standard simple model (Lab 8). It reads the prepared text from Part 4.

Records without a value for the target are left out, and the cell says how many. If the target cannot be used yet, for example because it has only one value, the cell skips the model and says why. **Macro F1** averages the F1 score of each class, so small classes count as much as large ones. It runs from 0 to 1; higher is better.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

baseline_scores = None
skip = ""
if not RUN_MS2_BASELINE:
    skip = "Part 7 is for MS2. Set RUN_MS2_BASELINE = True in Part 1.3 when you get there."
elif OWN_LABEL and TARGET not in data.columns:
    skip = "No labels loaded yet. Label at least 20 development records first (see Part 5.3)."
elif TARGET not in data.columns:
    skip = f"'{TARGET}' is not a column. Check TARGET in Part 1.3, or load your labels in Part 5.3."
else:
    # Only records with a value for the target can be used to fit or to score.
    has_target = data[TARGET].notna()
    train_with_target = data[has_target & data["split"].eq("train")]
    if not OWN_LABEL:
        print("records left out because the target is empty:", int((~has_target).sum()))
    if OWN_LABEL and len(train_with_target) < 20:
        skip = "Label at least 20 development records first."
    elif OWN_LABEL:
        # train_test_split shuffles the labeled records with the seed and keeps 30% apart for the check.
        fit_rows, check_rows = train_test_split(train_with_target, test_size=0.3, random_state=SEED)
        check_name = "development check (30% of your labels)"
    else:
        fit_rows = train_with_target
        check_rows = data[has_target & data["split"].eq("valid")]
        check_name = "valid"
    if not skip and fit_rows[TARGET].nunique() < 2:
        skip = "The fitting records have only one target value, so there is nothing to learn. Choose another target or subset."

if skip:
    print("Skipped:", skip)
else:
    check_protected(fit_rows)
    check_protected(check_rows)
    # The labels as text, so that number and word labels are handled the same way.
    y_fit = fit_rows[TARGET].astype(str)
    y_check = check_rows[TARGET].astype(str)
    # The majority baseline: the most common fitting label (mode()[0] takes the first one if tied).
    majority = y_fit.mode()[0]
    # Keep words and two-word phrases found in at least 5 texts (at least 2 with a small own-label sample).
    min_texts = 5
    if OWN_LABEL:
        min_texts = 2
    # make_pipeline chains TF-IDF and logistic regression; max_iter=1000 gives the solver enough steps to finish.
    pipeline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=min_texts),
                             LogisticRegression(max_iter=1000))
    # fit learns from the fitting records only; predict labels the check records.
    pipeline.fit(fit_rows["text_prepared"], y_fit)
    check_pred = pipeline.predict(check_rows["text_prepared"])
    # [majority] * len(check_rows) repeats the majority label once for every check record.
    baseline_scores = pd.DataFrame([
        {"method": f"majority ({majority})", "macro F1": f1_score(y_check, [majority] * len(check_rows), average="macro")},
        {"method": "TF-IDF + logistic regression", "macro F1": f1_score(y_check, check_pred, average="macro")},
    ]).round(3)
    baseline_scores["checked on"] = check_name
    baseline_scores["n checked"] = len(check_rows)
    baseline_scores.to_csv(OUT / "first_baseline.csv", index=False)
    print(baseline_scores.to_string(index=False), end="\n\n")
    # Precision and recall of each class for the TF-IDF model; recall is the share of a class that it finds.
    print(classification_report(y_check, check_pred, zero_division=0))

If the TF-IDF model is clearly above the majority guess, the target is predictable from the text in this sample. That does not yet show the method is good enough for the decision. Look at the recall of each class, and read some of the costly errors, as in Lab 8. If the two methods are close, check your labels and your question before you add a more complex method.

A star band is easy to predict from the words, because people write what they rate. A project needs a target closer to a decision, such as a problem type, an aspect or a review that needs a reply.

## Part 8 Save the outputs

### 8.1 Save the tables

Only tables without text or IDs go to the outputs folder. The sample, the ID lists and your label files are already in `project/data/`. The cell lists every file this run wrote.

In [ ]:
# The provenance as a two-column table (.T turns the one row into a column), then the audit tables.
pd.DataFrame([provenance]).T.rename(columns={0: "value"}).to_csv(OUT / "provenance.csv")
audit.to_csv(OUT / "audit_text_checks.csv", index=False)
length_table.to_csv(OUT / "audit_length_by_split.csv")
stars_table.to_csv(OUT / "audit_stars_by_split.csv")
concentration.to_csv(OUT / "audit_business_concentration.csv", index=False)
pii_table.to_csv(OUT / "audit_personal_details_scan.csv", index=False)
prep_audit.to_csv(OUT / "preparation_audit.csv", index=False)
# The sampler's filters and counts, without any text.
sample_counts = {"filters": sample_manifest["filters"], "counts": sample_manifest["counts"]}
(OUT / "sample_counts.json").write_text(json.dumps(sample_counts, indent=2))
# The files written in this run, for the manifest.
saved = [OUT / "provenance.csv", OUT / "audit_text_checks.csv", OUT / "audit_length_by_split.csv",
         OUT / "audit_stars_by_split.csv", OUT / "audit_business_concentration.csv",
         OUT / "audit_personal_details_scan.csv", OUT / "preparation_audit.csv", OUT / "sample_counts.json",
         OUT / "length_by_split.png", OUT / "evaluation_plan.json",
         SAMPLE_DIR / "protected_test_ids.csv", SAMPLE_DIR / "dropped_cross_split_copies.csv"]
if label_table is not None:
    label_table.to_csv(OUT / "audit_label_balance.csv")
    saved.append(OUT / "audit_label_balance.csv")
if baseline_scores is not None:
    saved.append(OUT / "first_baseline.csv")

### 8.2 Reproducibility record

The manifest is the record a reader uses to check a rerun. It holds your settings, the seed and the package versions. It also holds a SHA-256 hash of each input file, of your label files and of every output of this run. A **SHA-256 hash** is a 64-character code computed from a file's bytes; if one character changes, the hash changes. The **fingerprint** of an ID list is the hash of the sorted IDs; it shows that the protected test set never changed. Module 14 explains these records. You only need to run the two cells.

The sample hash is computed again on every run, so the record shows if the sample changed. Anyone with the same download can draw the same sample and check it.

In [ ]:
from datetime import datetime, timezone
import hashlib
import platform
import matplotlib
import sklearn

def text_sha256(path):
    """Take a file path; return the SHA-256 hash of the file's bytes (Windows line endings read as \n)."""
    file_bytes = Path(path).read_bytes().replace(b"\r\n", b"\n")
    return hashlib.sha256(file_bytes).hexdigest()

def ids_sha256(ids):
    """Take a list of IDs; return one fingerprint of the list (the order of the IDs does not matter)."""
    id_texts = []
    for record_id in ids:
        id_texts.append(str(record_id))
    joined = "\n".join(sorted(id_texts))
    return hashlib.sha256(joined.encode()).hexdigest()

def file_record(path):
    """Take a file path; return its path inside the repository and the first 12 characters of its hash."""
    path = Path(path)
    # relative_to(REPO) drops the "../.." part; as_posix() writes the path with / on every computer.
    return {"file": path.relative_to(REPO).as_posix(), "sha256": text_sha256(path)[:12]}

versions = {"python": platform.python_version(), "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__}

# The downloaded files (not committed) and the sample (committed), each with the start of its hash.
input_files = []
for f in sample_manifest["input_files"]:
    input_files.append({"file": f["file"], "sha256": f["sha256"][:12], "committed": False})
input_files.append({"file": "project/data/sample/yelp_sample.csv", "sha256": text_sha256(SAMPLE)[:12], "committed": True})

In [ ]:
# The label files, if you have them: their hashes show later that labels did not change after you saw results.
label_files = []
for path in [DEV_LABELS, HELDOUT_LABELS]:
    if path.exists():
        label_files.append(file_record(path))
output_records = []
for path in saved:
    output_records.append(file_record(path))
manifest = {
    "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "notebook": "yelp_reviews_project_starter",
    "settings": SETTINGS,
    "seed": SEED,
    "versions": versions,
    "input_files": input_files,
    "protected_test": {"n_ids": len(test_ids), "ids_sha256": ids_sha256(test_ids)},
    "heldout_sample": {"split": "valid", "n_ids": len(heldout_ids), "ids_sha256": ids_sha256(heldout_ids)},
    "label_files": label_files,
    "outputs": output_records,
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print("protected test fingerprint:", manifest["protected_test"]["ids_sha256"][:12])
print("files saved in this run:")
for record in output_records + [file_record(OUT / "manifest.json")]:
    print(" ", record["file"])

## Commit your work

1. Save this notebook (keep the outputs).
2. In GitHub Desktop, look at the changed files. You should see this notebook and the files in `project/outputs/option2_setup/`, your sample files in `project/data/sample/` and any label files in `project/data/annotations/`. You should **not** see anything from `project/data/private/`.
3. Write a short commit message, such as "MS1 setup: audit and protected split", then commit and push.
4. Push before the deadline. The instructor grades what your repository holds at the deadline, so a commit that is not pushed does not count.

## Next steps

- **MS1 (Week 6):** use the outputs above in your plan: the provenance record, the audit tables, the preparation check, the protected test and the evaluation plan. Say in your README how to download the data and give the sample settings; anyone can then draw the same sample.
- **MS2 (Week 9):** set `RUN_MS2_BASELINE = True` and build your baseline from Parts 5 and 7. Lab notebooks 8 and 9 show the classifier, the error review and the threshold steps.
- **MS3 (Week 13):** compare your selected method with the baseline once: on the protected test, or, with your own label, on the held-out sample. Report k of n and the paired comparison with its bootstrap interval from Lab 13.
- **Final project (Week 15):** Lab 14 adds an interval for each key number (Wilson for a share, a bootstrap interval for other metrics) and the exhibit standard.
- **Use TF-IDF as your text representation.** The saved course embeddings cover only the course data, and the course environment has no embedding model for new texts. TF-IDF (Labs 4, 6 and 8) is the supported representation.
- **An LLM is optional.** If you use one, save every output with the model, version, date and prompt. You are responsible for following the terms you accepted when you downloaded the data.